In [9]:
import pandas as pd
import numpy as np
import splitting
import tuning as tn

In [10]:
df = tn.load_tuning_data()
feats = tn.load_feature_set("pruned", df)
print(df.shape, len(feats))

(4510, 123) 79


In [3]:
import json

with open("model_cards.json") as f:
    cards = json.load(f)

print(json.dumps(cards, indent=2)[:3000])

{
  "pm25_reg_t1_xgboost": {
    "task": "regression",
    "horizon_days": 1,
    "model": "xgboost",
    "features": [
      "uv_index",
      "s5p_no2_col",
      "s5p_o3_col",
      "s5p_no2_col_gap_days",
      "road_length_km_3000",
      "street_length_avg_3000",
      "circuity_avg_3000",
      "major_road_km_3000",
      "major_road_share_5000",
      "industrial_count_5000",
      "temperature_2m_mean",
      "temperature_2m_max",
      "temperature_2m_min",
      "temperature_2m_std",
      "relative_humidity_2m_mean",
      "relative_humidity_2m_max",
      "relative_humidity_2m_min",
      "relative_humidity_2m_std",
      "surface_pressure_mean",
      "surface_pressure_max",
      "surface_pressure_min",
      "surface_pressure_std",
      "precipitation_mean",
      "precipitation_max",
      "precipitation_std",
      "wind_speed_10m_mean",
      "wind_speed_10m_max",
      "wind_speed_10m_min",
      "wind_speed_10m_std",
      "is_weekend",
      "month",
      "is_Ra

In [4]:
for name, card in cards.items():
    if "clf" in name:
        print(name)
        print({k: v for k, v in card.items() if k not in ("features", "pkl_sha256")})
        print()

pm25_clf_t1_xgboost
{'task': 'health-risk classification', 'horizon_days': 1, 'model': 'xgboost', 'n_features': 79, 'training_rows': 4510, 'training_window': ['2023-09-08', '2026-02-25'], 'protocol_version': 'w4-v1', 'onnx': True, 'feature_set': 'pruned', 'params': {'max_depth': 7, 'learning_rate': 0.014763831389514571, 'n_estimators': 250, 'min_child_weight': 8.387197772688628, 'subsample': 0.5878664953062325, 'colsample_bytree': 0.893800119143862, 'gamma': 0.00042118445581121165, 'reg_lambda': 0.004964303760846287}, 'imbalance': 'none', 'alert_threshold': 0.60082895, 'alert_rule': 'predict_proba(X)[:,1] >= alert_threshold', 'task4_cv_recall_at_precision75_mean': 0.3633967633714789, 'task4_cv_recall_std': 0.3405924511025518, 'libs': {'python': '3.12.3', 'numpy': '2.4.4', 'pandas': '3.0.2', 'lightgbm': '4.7.0', 'xgboost': '3.4.1'}}

pm25_clf_t2_xgboost
{'task': 'health-risk classification', 'horizon_days': 2, 'model': 'xgboost', 'n_features': 79, 'training_rows': 4510, 'training_window

In [11]:
clf = {
    1: dict(params={'max_depth': 7, 'learning_rate': 0.014763831389514571, 'n_estimators': 250,
                    'min_child_weight': 8.387197772688628, 'subsample': 0.5878664953062325,
                    'colsample_bytree': 0.893800119143862, 'gamma': 0.00042118445581121165,
                    'reg_lambda': 0.004964303760846287}, imbalance="none"),
    2: dict(params={'max_depth': 7, 'learning_rate': 0.024413410974874053, 'n_estimators': 100,
                    'min_child_weight': 14.792172339404473, 'subsample': 0.78695602799705,
                    'colsample_bytree': 0.7771653661840072, 'gamma': 0.014245783828916288,
                    'reg_lambda': 0.09831126761740053}, imbalance="class_weight"),
    3: dict(params={'max_depth': 5, 'learning_rate': 0.07049823976311798, 'n_estimators': 350,
                    'min_child_weight': 12.043333438925757, 'subsample': 0.5893757128485898,
                    'colsample_bytree': 0.9861633529913195, 'gamma': 0.000908674754602063,
                    'reg_lambda': 0.4675099787826079}, imbalance="class_weight"),
}

oof = {}
for h in (1, 2, 3):
    res = tn.cv_evaluate("xgboost", clf[h]["params"], df, h, feats, task="clf",
                         imbalance=clf[h]["imbalance"], return_oof=True)
    oof[h] = res.oof
    print(f"horizon {h}: rows={len(res.oof)}  risk-day share={res.oof.y_true.mean():.3f}  "
          f"recall@precision0.75={res.score:.3f}")

horizon 1: rows=2250  risk-day share=0.269  recall@precision0.75=0.345
horizon 2: rows=2250  risk-day share=0.268  recall@precision0.75=0.115
horizon 3: rows=2250  risk-day share=0.268  recall@precision0.75=0.128


In [12]:

def at_precision(o, floor):
    p, r, t = precision_recall_curve(o.y_true, o.y_pred)
    ok = p[:-1] >= floor
    if not ok.any():
        return None
    return t[np.flatnonzero(ok)[np.argmax(r[:-1][ok])]]

In [16]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve


rows = []
for h in (1, 2, 3):
    o = oof[h]
    for floor in (0.75, 0.70, 0.65, 0.60, 0.55, 0.50):
        thr = at_precision(o, floor)
        if thr is None:
            continue
        alert = o.y_pred >= thr
        tp = int((alert & (o.y_true == 1)).sum())
        fp = int((alert & (o.y_true == 0)).sum())
        rows.append(dict(horizon=h, precision_floor=floor, threshold=thr,
                         recall=tp / (o.y_true == 1).sum(),
                         precision=tp / max(alert.sum(), 1),
                         alerts_per_100=100 * alert.mean(),
                         false_alarms_per_100=100 * fp / len(o)))

tradeoff = pd.DataFrame(rows).round(3)
tradeoff

,horizon,precision_floor,threshold,recall,precision,alerts_per_100,false_alarms_per_100
0,1,0.75,0.570,0.466,0.750,16.711,4.178
1,1,0.70,0.445,0.613,0.700,23.556,7.067
2,1,0.65,0.358,0.707,0.650,29.244,10.222
3,1,0.60,0.298,0.754,0.600,33.778,13.511
4,1,0.55,0.228,0.810,0.551,39.556,17.778
5,1,0.50,0.173,0.858,0.500,46.133,23.067
6,2,0.75,0.839,0.108,0.756,3.822,0.933
7,2,0.70,0.763,0.290,0.700,11.111,3.333
8,2,0.65,0.695,0.447,0.651,18.444,6.444
9,2,0.60,0.639,0.536,0.600,24.000,9.600


In [7]:
tradeoff.to_csv("tradeoff.csv")

In [8]:
chosen = {1: 0.65, 2: 0.55, 3: 0.50}
season = {1: "winter", 2: "spring", 3: "summer", 4: "autumn", 5: "winter"}

for h, floor in chosen.items():
    thr = at_precision(oof[h], floor)
    print(f"\nhorizon {h}: precision floor {floor}, threshold {thr:.3f}")
    out = []
    for k, g in oof[h].groupby("fold"):
        a = g.y_pred >= thr
        tp = int((a & (g.y_true == 1)).sum())
        out.append(dict(fold=k, season=season[k], risk_days=int(g.y_true.sum()),
                        alerts=int(a.sum()), caught=tp,
                        recall=round(tp / max(g.y_true.sum(), 1), 2),
                        precision=round(tp / max(a.sum(), 1), 2)))
    print(pd.DataFrame(out).to_string(index=False))


horizon 1: precision floor 0.65, threshold 0.358
 fold season  risk_days  alerts  caught  recall  precision
    1 winter        180     217     144    0.80       0.66
    2 spring        113     112      61    0.54       0.54
    3 summer         32      16       6    0.19       0.38
    4 autumn         49      51      27    0.55       0.53
    5 winter        231     262     190    0.82       0.73

horizon 2: precision floor 0.55, threshold 0.559
 fold season  risk_days  alerts  caught  recall  precision
    1 winter        179     295     160    0.89       0.54
    2 spring        113      90      39    0.35       0.43
    3 summer         32       0       0    0.00       0.00
    4 autumn         52      45      13    0.25       0.29
    5 winter        228     273     175    0.77       0.64

horizon 3: precision floor 0.5, threshold 0.414
 fold season  risk_days  alerts  caught  recall  precision
    1 winter        179     302     151    0.84       0.50
    2 spring        111  

In [9]:
winter_months = {12, 1, 2}
rows = []
for h in (1, 2, 3):
    o = oof[h].copy()
    o["winter"] = df.iloc[o.row.values]["month"].isin(winter_months).values
    for name, g in (("winter", o[o.winter]), ("not winter", o[~o.winter])):
        for floor in (0.60, 0.50, 0.40):
            thr = at_precision(g, floor)
            if thr is None:
                continue
            a = g.y_pred >= thr
            tp = int((a & (g.y_true == 1)).sum())
            rows.append(dict(horizon=h, group=name, risk_days=int(g.y_true.sum()),
                             floor=floor, threshold=round(thr, 3),
                             recall=round(tp / g.y_true.sum(), 2),
                             precision=round(tp / max(a.sum(), 1), 2),
                             alerts_per_100=round(100 * a.mean(), 1)))
print(pd.DataFrame(rows).to_string(index=False))

 horizon      group  risk_days  floor  threshold  recall  precision  alerts_per_100
       1     winter        405    0.6      0.201    0.93       0.60            71.8
       1     winter        405    0.5      0.075    1.00       0.50            92.6
       1     winter        405    0.4      0.023    1.00       0.46           100.0
       1 not winter        200    0.6      0.571    0.27       0.60             6.5
       1 not winter        200    0.5      0.343    0.50       0.50            14.4
       1 not winter        200    0.4      0.205    0.64       0.40            23.4
       2     winter        404    0.6      0.566    0.81       0.60            62.4
       2     winter        404    0.5      0.286    0.99       0.50            91.7
       2     winter        404    0.4      0.141    1.00       0.46           100.0
       2 not winter        200    0.6      0.791    0.01       0.67             0.2
       2 not winter        200    0.5      0.755    0.04       0.50         

In [10]:
import json

chosen = {1: 0.65, 2: 0.55, 3: 0.50}            # precision level for the "warning" alert
HIGH = {1: 0.601, 2: 0.804, 3: 0.866}           # current thresholds, unchanged

cfg = {str(h): {"warning": round(float(at_precision(oof[h], chosen[h])), 3), "high": HIGH[h]}
       for h in (1, 2, 3)}

json.dump(cfg, open(tn.ART_DIR / "alert_config.json", "w"), indent=2)
tradeoff.to_csv(tn.ART_DIR / "alert_tradeoff_table.csv", index=False)
print(json.dumps(cfg, indent=2))

{
  "1": {
    "warning": 0.358,
    "high": 0.601
  },
  "2": {
    "warning": 0.559,
    "high": 0.804
  },
  "3": {
    "warning": 0.414,
    "high": 0.866
  }
}


In [17]:
import json
raw = json.load(open("task2_best_lightgbm_params.json"))
lgbm_params = {int(k[2:]): v["best_params"] for k, v in raw.items()}

xgb_t1 = {'max_depth': 7, 'learning_rate': 0.011459387083522775, 'n_estimators': 650,
          'min_child_weight': 3.9147095640372895, 'subsample': 0.519168947898092,
          'colsample_bytree': 0.43730447808879536, 'gamma': 8.515493263455816e-06,
          'reg_lambda': 0.008233393478614112}

final_reg = {1: ("xgboost", xgb_t1),
             2: ("lightgbm", {**lgbm_params[2], "subsample_freq": 1}),
             3: ("lightgbm", {**lgbm_params[3], "subsample_freq": 1})}

sat = ["s5p_no2_col", "s5p_o3_col", "s5p_no2_col_gap_days"]
no_sat = [f for f in feats if f not in sat]

print("REGRESSION (RMSE)")
for h, (m, p) in final_reg.items():
    a = tn.cv_evaluate(m, p, df, h, feats, task="reg")
    b = tn.cv_evaluate(m, p, df, h, no_sat, task="reg")
    print(f"t+{h} {m}: with satellite {a.score:.3f} | without {b.score:.3f}")

print("\nALERT MODEL (AUC)")
for h in (1, 2, 3):
    a = tn.cv_evaluate("xgboost", clf[h]["params"], df, h, feats, task="clf", imbalance=clf[h]["imbalance"])
    b = tn.cv_evaluate("xgboost", clf[h]["params"], df, h, no_sat, task="clf", imbalance=clf[h]["imbalance"])
    print(f"t+{h}: with satellite {a.summary().loc['auc','mean']:.3f} | without {b.summary().loc['auc','mean']:.3f}")

REGRESSION (RMSE)
t+1 xgboost: with satellite 14.868 | without 14.921
t+2 lightgbm: with satellite 16.909 | without 16.960
t+3 lightgbm: with satellite 17.533 | without 17.431

ALERT MODEL (AUC)
t+1: with satellite 0.806 | without 0.806
t+2: with satellite 0.716 | without 0.724
t+3: with satellite 0.638 | without 0.648


In [7]:
import os, joblib, numpy as np

os.makedirs("models_v2", exist_ok=True)
X = df[no_sat]
print("features:", len(no_sat))          

# 3 regression models (price forecast)
for h, (m, p) in final_reg.items():
    model = tn.build_model(m, "reg", p).fit(X, df[f"target_{h}d"])
    joblib.dump(model, f"models_v2/pm25_reg_t{h}_{m}.pkl")
    print("saved regression", h, m)

# 3 alert models (risk probability)
for h in (1, 2, 3):
    y = df[f"target_risk_{h}d"].to_numpy()
    fit_kw = {}
    if clf[h]["imbalance"] == "class_weight":
        n, pos = len(y), float(y.sum())
        fit_kw["sample_weight"] = np.where(y == 1, n / (2 * pos), n / (2 * (n - pos)))
    model = tn.build_model("xgboost", "clf", clf[h]["params"]).fit(X, y, **fit_kw)
    joblib.dump(model, f"models_v2/pm25_clf_t{h}_xgboost.pkl")
    print("saved classifier", h)

json.dump(no_sat, open("models_v2/features_76.json", "w"), indent=2)

features: 76
saved regression 1 xgboost
saved regression 2 lightgbm
saved regression 3 lightgbm
saved classifier 1
saved classifier 2
saved classifier 3


In [18]:
oof76 = {}
for h in (1, 2, 3):
    res = tn.cv_evaluate("xgboost", clf[h]["params"], df, h, no_sat,
                         task="clf", imbalance=clf[h]["imbalance"], return_oof=True)
    oof76[h] = res.oof
    print("done", h)

chosen = {1: 0.65, 2: 0.55, 3: 0.50}      # the precision we accept for the warning level
for h in (1, 2, 3):
    t = at_precision(oof76[h], chosen[h])
    o = oof76[h]
    flag = o.y_pred >= t
    prec = o.y_true[flag].mean()
    rec = flag[o.y_true == 1].mean()
    print(f"t+{h}: warning threshold {t:.3f} | precision {prec:.2f} | recall {rec:.2f}")

done 1
done 2
done 3
t+1: warning threshold 0.362 | precision 0.65 | recall 0.70
t+2: warning threshold 0.557 | precision 0.55 | recall 0.65
t+3: warning threshold 0.445 | precision 0.50 | recall 0.57


In [19]:
import json

cfg = {}
for h in (1, 2, 3):
    cfg[str(h)] = {
        "warning": round(float(at_precision(oof76[h], chosen[h])), 3),
        "high":    round(float(at_precision(oof76[h], 0.75)), 3),
    }
print(json.dumps(cfg, indent=2))
json.dump(cfg, open("models_v2/alert_config.json", "w"), indent=2)

{
  "1": {
    "warning": 0.362,
    "high": 0.587
  },
  "2": {
    "warning": 0.557,
    "high": 0.802
  },
  "3": {
    "warning": 0.445,
    "high": 0.863
  }
}


In [20]:
import json
f = json.load(open("models_v2/features_76.json"))
for i in range(0, len(f), 4):
    print(", ".join(f[i:i+4]))
print(df[["station", "date"]].head(3) if "station" in df and "date" in df else df.columns[:12].tolist())

uv_index, road_length_km_3000, street_length_avg_3000, circuity_avg_3000
major_road_km_3000, major_road_share_5000, industrial_count_5000, temperature_2m_mean
temperature_2m_max, temperature_2m_min, temperature_2m_std, relative_humidity_2m_mean
relative_humidity_2m_max, relative_humidity_2m_min, relative_humidity_2m_std, surface_pressure_mean
surface_pressure_max, surface_pressure_min, surface_pressure_std, precipitation_mean
precipitation_max, precipitation_std, wind_speed_10m_mean, wind_speed_10m_max
wind_speed_10m_min, wind_speed_10m_std, is_weekend, month
is_Ramadan, is_Khamsin_season, pm10_lag_1d, pm10_lag_2d
pm10_lag_3d, pm10_lag_7d, pm2_5_corrected_lag_1d, pm2_5_corrected_lag_2d
pm2_5_corrected_lag_3d, pm2_5_corrected_lag_7d, carbon_monoxide_lag_1d, carbon_monoxide_lag_2d
carbon_monoxide_lag_3d, carbon_monoxide_lag_7d, nitrogen_dioxide_lag_1d, nitrogen_dioxide_lag_2d
nitrogen_dioxide_lag_3d, nitrogen_dioxide_lag_7d, sulphur_dioxide_lag_1d, sulphur_dioxide_lag_2d
sulphur_dioxide_

In [22]:
static = ["road_length_km_3000","street_length_avg_3000","circuity_avg_3000",
          "major_road_km_3000","major_road_share_5000","industrial_count_5000",
          "station_id_encoded"]
tbl = df.groupby("station_id")[static].first()
tbl.to_csv("models_v2/station_static.csv")
print(tbl)

            road_length_km_3000  street_length_avg_3000  circuity_avg_3000  \
station_id                                                                   
downtown               1404.083                   46.56             1.0304   
helwan                  875.980                   50.54             1.0243   
new_cairo               506.892                   93.73             1.0616   
october                 664.630                   93.82             1.0335   
shobra                 1533.607                   47.31             1.0168   

            major_road_km_3000  major_road_share_5000  industrial_count_5000  \
station_id                                                                     
downtown               143.803                 0.0850                   35.0   
helwan                  12.742                 0.0434                  105.0   
new_cairo               81.239                 0.1336                   40.0   
october                 17.055                 0.0299

In [24]:
import sys; sys.path.insert(0, "pipeline")
import importlib, pipeline.features as fe; importlib.reload(fe)

t = "2025-11-12"
start = (pd.Timestamp(t) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
hourly = {s: fe.fetch_hourly(s, start, t) for s in fe.COORDS}
new = fe.build_features(hourly, t)

day = pd.to_datetime(df["day"]).dt.tz_localize(None)
old = df[day == t].set_index("station_id").loc[new.index, new.columns].astype(float)

bad = ~np.isclose(new, old, rtol=1e-3, atol=1e-3)
print("shape:", new.shape, "| mismatching values:", int(bad.sum()), "of", bad.size)
print(pd.DataFrame({"mismatches": bad.sum(0), "max_diff": (new - old).abs().max()})
        .query("mismatches > 0").to_string())

shape: (5, 76) | mismatching values: 38 of 380
                             mismatches   max_diff
temperature_2m_mean                   5   0.029167
temperature_2m_max                    2   0.050000
temperature_2m_min                    1   0.050000
temperature_2m_std                    2   0.012412
relative_humidity_2m_max              3   0.467060
relative_humidity_2m_min              5   0.396260
relative_humidity_2m_std              4   0.075074
surface_pressure_std                  2   0.007876
wind_speed_10m_max                    3   0.048061
wind_speed_10m_min                    3   0.043053
wind_speed_10m_std                    2   0.009212
sulphur_dioxide_lag_7d                1  36.348611
sulphur_dioxide_roll_avg_7d           2   9.166071
inversion_proxy                       1   8.676675
inversion_proxy_v2                    2   0.103372


In [25]:
!python pipeline/run_forecast.py

  station issue_date target_date  horizon  pred_pm25  risk_prob alert_level                   aqi_category
 downtown 2026-11-22  2026-11-23        1       50.9      0.383     warning Unhealthy for Sensitive Groups
   helwan 2026-11-22  2026-11-23        1       47.3      0.185        none Unhealthy for Sensitive Groups
   shobra 2026-11-22  2026-11-23        1       54.7      0.403     warning Unhealthy for Sensitive Groups
new_cairo 2026-11-22  2026-11-23        1       38.3      0.064        none Unhealthy for Sensitive Groups
  october 2026-11-22  2026-11-23        1       37.0      0.129        none Unhealthy for Sensitive Groups
 downtown 2026-11-22  2026-11-24        2       49.8      0.350        none Unhealthy for Sensitive Groups
   helwan 2026-11-22  2026-11-24        2       50.7      0.404        none Unhealthy for Sensitive Groups
   shobra 2026-11-22  2026-11-24        2       52.0      0.452        none Unhealthy for Sensitive Groups
new_cairo 2026-11-22  2026-11-24     